# Task 4: General Health Query Chatbot

## Objective

The objective of this project is to build a health information chatbot using Google's Gemini API.

The chatbot:
- Answers general health questions
- Uses prompt engineering
- Includes safety filters
- Provides educational information only

## Environment setup

We configure API access securely using .env and install required packages.

In [ ]:
# Install required libraries
!pip install google-generativeai python-dotenv

## Dependencies and imports

Install and import the packages required for Gemini and dotenv.

In [ ]:
# Import required modules
import os
import re
from pathlib import Path
from dotenv import load_dotenv
import google.generativeai as genai

## Load credentials

We load the Gemini API key from a `.env` file to avoid hardcoding sensitive credentials.

In [ ]:
# Load environment variables
dotenv_path = Path("e:/AI-ML-Internship/task4_health_chatbot/.env")
if not dotenv_path.is_file():
    raise FileNotFoundError(f".env file not found at {dotenv_path}")

loaded = load_dotenv(str(dotenv_path), override=True)
if not loaded:
    raise RuntimeError(f"Unable to load environment variables from {dotenv_path}")

api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError(
        "GEMINI_API_KEY is missing. Add it to .env and restart the notebook kernel."
    )

genai.configure(api_key=api_key)

## Initialize Gemini

We use `models/gemini-2.5-flash` for fast and efficient responses.

In [ ]:
# Initialize the Gemini model
model = genai.GenerativeModel("models/gemini-2.5-flash")

## Prompt design

We prepare a prompt that keeps the assistant friendly, simple, and safe.

In [ ]:
# System-style prompt to guide model behavior
BASE_PROMPT = """
You are a friendly health advisor.
Answer the user's health question in a warm, clear, and conversational tone.
Keep the response simple and easy to read.
Do not use markdown headings, bold text, italics, bullet symbols, or code formatting.
Do not format the answer as a markdown list with symbols such as *, -, +, or numbered items.
Do not recommend prescriptions, dosages, or medical diagnoses.
Always encourage the user to consult a qualified healthcare professional when appropriate.

User question:
"""

## Safety filter

We add a basic rule-based filter to block unsafe medical requests such as dosage or prescriptions.

In [ ]:
# Keywords that indicate unsafe medical advice requests
unsafe_keywords = [
    "dose", "dosage", "prescribe", "medicine",
    "drug", "antibiotic", "treatment plan"
]

def is_safe_query(query):
    """
    Checks if user query is safe to process.
    Returns False if it contains restricted medical advice requests.
    """
    query_lower = query.lower()
    
    for word in unsafe_keywords:
        if word in query_lower:
            return False
    
    return True

## Chatbot implementation

Defines model output cleanup and the chatbot function.

In [ ]:
def clean_response_text(text):
    """Clean the model output and remove markdown formatting."""
    if not text:
        return "No response was returned by the model."

    text = re.sub(r'(?m)^\s*#{1,6}\s*', '', text)
    text = re.sub(r'\*\*(.*?)\*\*', r'\1', text)
    text = re.sub(r'__(.*?)__', r'\1', text)
    text = re.sub(r'\*(.*?)\*', r'\1', text)
    text = re.sub(r'(?m)^\s*\d+\.\s*', '', text)
    text = re.sub(r'(?m)^[\-\*\+]\s*', '', text)
    text = re.sub(r'\[(.*?)\]\((.*?)\)', r'\1', text)

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return "\n".join(lines)


def health_chatbot(user_query):
    """Respond to general health questions safely and clearly."""

    if not is_safe_query(user_query):
        return (
            "I can provide general health information, but I cannot recommend medicines, dosages, or treatment plans. "
            "Please consult a qualified healthcare professional for specific advice."
        )

    prompt = BASE_PROMPT + user_query
    response = model.generate_content(prompt)
    response_text = getattr(response, "text", None)
    return clean_response_text(response_text)

## Example usage

Test the chatbot with general health queries and restricted medical questions.

In [ ]:
def display_chatbot_response(user_query):
    answer = health_chatbot(user_query)
    print(f"Question: {user_query}\n")
    print("Answer:")
    print(answer)
    print()

# Example 1: General symptom question
display_chatbot_response("What causes a sore throat?")

# Example 2: Another general health question
display_chatbot_response("Why do people get headaches?")

# Example 3: Restricted medical advice request
display_chatbot_response("What is the dosage of paracetamol for a child?")

## Step 9: Conclusion

### Key Learnings:
- Prompt engineering for controlled LLM responses
- API integration using Google Gemini
- Implementing safety filters in AI systems
- Building a simple conversational AI chatbot

---

### Outcome:
A functional and safe health chatbot capable of answering general medical questions responsibly.